# Formal daily chicken-heart replot

This notebook redraws the new daily interpolation outputs using the visual style of
`notebooks/heart` and `downstream_helpers/heart.py`, while keeping the formal
`chicken_heart_analysis` outputs and formal model logic as the source of truth.

Rules used here:
- Input slices come from `chicken_heart_analysis/new_runs_formal_trained/formal_daily_piecewise_interpolation_celltypecorrected`.
- Side-by-side layout and stackbar style follow `downstream_helpers/heart.py`.
- Velocity components are recomputed from the formal trained model on each slice.
- If helper logic conflicts with formal logic, the formal logic wins.


In [ ]:
from __future__ import annotations

import json
import io
import logging
import sys
from contextlib import redirect_stderr
from pathlib import Path

from IPython.display import SVG, display
import anndata as ad
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scvelo as scv
import seaborn as sns
import torch

import os
REPO_ROOT = Path(os.environ.get("CYTOBRIDGE_SOURCE_DIR", ".")).resolve()
PROJECT_DIR = Path(os.environ.get("CYTOBRIDGE_PROJECT_DIR", ".")).resolve()
DATA_DIR = PROJECT_DIR / "data" / "chicken_heart"
sys.path.insert(0, str(REPO_ROOT / "reproduction" / "chicken_heart"))
ANALYSIS_ROOT = Path(os.environ.get("CYTOBRIDGE_HEART_OUTPUT_DIR", PROJECT_DIR / "outputs" / "chicken_heart_paper")).resolve()
CELLTYPE_SHARE_ROOT = DATA_DIR
import CytoBridge as cb
PACKAGE_ROOT = Path(cb.__file__).resolve().parents[1]
HELPER_ROOT = REPO_ROOT / "reproduction" / "chicken_heart" / "downstream_helpers"

for path in (REPO_ROOT, PACKAGE_ROOT, HELPER_ROOT):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import CytoBridge as cb
from downstream_helpers.heart import (
    HEART_LABEL_TO_COLOR,
    plot_celltype_stackbar,
    plot_segment_network_evolution,
    plot_side_by_side_spatial_with_custom_colors,
    prepare_data_for_side_by_side_2d,
)
from downstream_helpers.heart_lineage_functions import (
    plot_lineage_transition,
    prepare_multi_timepoint_adata,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
scv.settings.set_figure_params("scvelo")
sns.set_theme(style="white")
plt.rcParams["font.family"] = "DejaVu Sans"
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)

INTERP_RUN_DIR = ANALYSIS_ROOT / "new_runs_formal_trained" / "formal_daily_piecewise_interpolation_celltypecorrected"
SLICE_DIR = INTERP_RUN_DIR / "slice_data"
MANIFEST_PATH = INTERP_RUN_DIR / "manifest.json"
ALIGNED_H5AD_PATH = DATA_DIR / "aligned.h5ad"
MODEL_DIR = DATA_DIR / "model"
EDGE_PREDICTOR_PATH = DATA_DIR / "edge_classifier" / "chicken_heart_edge_model.pt"
CLASSIFIER_CACHE_PATH = DATA_DIR / "classifier_cache" / "classifier_resmlp_432f09f20ff65c0d.pt"
if not CLASSIFIER_CACHE_PATH.is_file():
    raise FileNotFoundError(CLASSIFIER_CACHE_PATH)
OUTPUT_DIR = ANALYSIS_ROOT / "new_runs_formal_trained" / "formal_daily_piecewise_replot_celltypecorrected"
SIDE_DIR = OUTPUT_DIR / "side_by_side_outputs"
VELOCITY_DIR = OUTPUT_DIR / "velocity_streams"
COMMUNICATION_DIR = OUTPUT_DIR / "communication_network"
LINEAGE_DIR = OUTPUT_DIR / "lineage_outputs"
SAVED_ANDATA_DIR = SIDE_DIR / "saved_anndata"

for path in (OUTPUT_DIR, SIDE_DIR, VELOCITY_DIR, COMMUNICATION_DIR, LINEAGE_DIR, SAVED_ANDATA_DIR):
    path.mkdir(parents=True, exist_ok=True)

with MANIFEST_PATH.open("r", encoding="utf-8") as handle:
    manifest = json.load(handle)

print(f"Using device: {DEVICE}")
print(f"Interpolation run dir: {INTERP_RUN_DIR}")
print(f"Corrected package root: {PACKAGE_ROOT}")
print(f"Corrected classifier cache: {CLASSIFIER_CACHE_PATH}")
print(f"Replot output dir: {OUTPUT_DIR}")

LEGACY_REGION_COLOR_MAP = {
    "Ventricle": "#7f7f7f",
    "Outflow tract": "#66c2a5",
    "Endothelium": "#9edae5",
    "Valves": "#c25bac",
    "Atria": "#cab2d6",
    "Epicardium": "#dbdb8d",
    "Trabecular LV and endocardium": "#4575b4",
    "Compact LV and inter-ventricular septum": "#aec7e8",
    "Right ventricle": "#c49c94",
}

In [ ]:
def canonical_label(value: object) -> str:
    return str(value).replace("\n", " ").replace("\r", " ").replace("  ", " ").strip()


def model_time_from_day_label(label: str) -> float:
    lookup = {"D4": 0.0, "D7": 1.0, "D10": 2.0, "D14": 3.0}
    text = canonical_label(label)
    if text not in lookup:
        raise KeyError(f"Unsupported observed time label for classifier training: {text}")
    return float(lookup[text])


def to_dense_float32(matrix: object) -> np.ndarray:
    if hasattr(matrix, "toarray"):
        matrix = matrix.toarray()
    return np.asarray(matrix, dtype=np.float32)


def load_daily_slice_dict(manifest_dict: dict, slice_dir: Path, manifest_key: str) -> tuple[dict[str, ad.AnnData], dict[str, float], dict[str, str | None]]:
    adata_dict: dict[str, ad.AnnData] = {}
    label_to_model_time: dict[str, float] = {}
    label_source_by_time: dict[str, str | None] = {}
    for item in sorted(manifest_dict["slices"], key=lambda row: float(row["time_float"])):
        time_label = str(item["time_label"])
        adata_path = Path(item[manifest_key])
        if not adata_path.is_absolute():
            adata_path = slice_dir / adata_path.name
        adata_t = ad.read_h5ad(adata_path)

        if "spatial" not in adata_t.obsm:
            if adata_t.X.shape[1] >= 2:
                adata_t.obsm["spatial"] = np.asarray(adata_t.X[:, :2], dtype=np.float32)
            else:
                raise ValueError(f"Slice {time_label} does not contain 2D spatial coordinates.")

        label_col = "celltype_prediction" if "celltype_prediction" in adata_t.obs.columns else None

        region_col = None
        for candidate in ("region", "Region", "anatomic_region"):
            if candidate in adata_t.obs.columns:
                region_col = candidate
                break

        if label_col is None:
            raise KeyError(
                f"Slice {time_label} is missing obs['celltype_prediction']; found columns: {list(adata_t.obs.columns)}"
            )
        labels = adata_t.obs[label_col].astype(str).map(canonical_label)
        adata_t.obs["celltype_prediction"] = labels.values
        adata_t.obs["annotation"] = labels.values
        if region_col is not None:
            adata_t.obs["region"] = adata_t.obs[region_col].astype(str).map(canonical_label).values
        adata_t.obs["timepoint"] = time_label
        adata_t.obs["time_label"] = time_label
        adata_t.obs["time_float"] = float(item["time_float"])
        adata_t.obs["is_observed"] = bool(item["is_observed"])
        adata_t.uns["time_label"] = time_label
        adata_t.uns["time_float"] = float(item["time_float"])
        adata_t.uns["is_observed"] = bool(item["is_observed"])
        adata_t.uns["slice_origin"] = item["slice_origin"]
        adata_t.uns["source_anchor_time"] = float(item["source_anchor_time"])

        adata_dict[time_label] = adata_t
        label_to_model_time[time_label] = float(item["time_float"])
        label_source_by_time[time_label] = label_col
    return adata_dict, label_to_model_time, label_source_by_time


def attach_region_to_observed_slices(adata_dict: dict[str, ad.AnnData], reference_h5ad_path: Path) -> None:
    reference = ad.read_h5ad(reference_h5ad_path)
    region_col = None
    for candidate in ("region", "Region", "anatomic_region"):
        if candidate in reference.obs.columns:
            region_col = candidate
            break
    if region_col is None:
        raise KeyError(f"Reference aligned h5ad is missing region columns. Found: {list(reference.obs.columns)}")

    region_lookup = reference.obs[region_col].astype(str).map(canonical_label)
    for time_label, adata_t in adata_dict.items():
        if not bool(adata_t.uns.get("is_observed", False)):
            continue
        shared = adata_t.obs_names.intersection(region_lookup.index)
        if len(shared) == 0:
            print(f"Warning: no shared obs_names found to attach region for {time_label}.")
            continue
        region_series = pd.Series(index=adata_t.obs_names, dtype=object)
        region_series.loc[shared] = region_lookup.loc[shared].values
        missing = int(region_series.isna().sum())
        if missing > 0:
            print(f"Warning: {missing} observed cells in {time_label} missing region after join; filling with 'Unknown'.")
        adata_t.obs["region"] = region_series.fillna("Unknown").astype(str).map(canonical_label).values


def build_label_palette(adata_dict: dict[str, ad.AnnData]) -> dict[str, str]:
    labels = sorted({canonical_label(v) for adata_t in adata_dict.values() for v in adata_t.obs["celltype_prediction"].astype(str)})
    base = {canonical_label(label): str(color) for label, color in HEART_LABEL_TO_COLOR.items()}
    missing = [label for label in labels if label not in base]
    if missing:
        extra = sns.color_palette("husl", len(missing)).as_hex()
        for label, color in zip(missing, extra):
            base[label] = mcolors.to_hex(color)
    return {label: base[label] for label in labels}


def build_region_palette(adata_dict: dict[str, ad.AnnData]) -> dict[str, str]:
    regions = sorted({canonical_label(v) for adata_t in adata_dict.values() if "region" in adata_t.obs.columns for v in adata_t.obs["region"].astype(str)})
    base = {canonical_label(label): str(color) for label, color in LEGACY_REGION_COLOR_MAP.items()}
    missing = [region for region in regions if region not in base]
    if missing:
        extra = sns.color_palette("husl", len(missing)).as_hex()
        for region, color in zip(missing, extra):
            base[region] = mcolors.to_hex(color)
    return {region: base[region] for region in regions}


daily_adata_dict, label_to_model_time, label_source_by_time = load_daily_slice_dict(manifest, SLICE_DIR, "slice_h5ad")
daily_comm_adata_dict, _, comm_label_source_by_time = load_daily_slice_dict(manifest, SLICE_DIR, "communication_h5ad")
attach_region_to_observed_slices(daily_adata_dict, ALIGNED_H5AD_PATH)
time_keys = list(daily_adata_dict.keys())
observed_time_keys = [key for key in time_keys if bool(daily_adata_dict[key].uns.get("is_observed", False))]
label_to_color = build_label_palette(daily_adata_dict)
region_to_color = build_region_palette(daily_adata_dict)

print("Daily time keys:", time_keys)
print("Observed time keys:", observed_time_keys)
print("Stored label-source columns by time:", label_source_by_time)
print("Communication label-source columns by time:", comm_label_source_by_time)
print("Palette labels:", sorted(label_to_color))
print("Region labels:", sorted(region_to_color))

combined_timeline = ad.concat(
    [daily_adata_dict[key] for key in time_keys],
    join="outer",
    label="time_label",
    keys=time_keys,
    index_unique="__",
)
combined_timeline.write_h5ad(SAVED_ANDATA_DIR / "daily_real_plus_interpolated_all_times.h5ad")
combined_timeline.obs.to_csv(SAVED_ANDATA_DIR / "daily_real_plus_interpolated_all_times.csv")

In [ ]:
side_by_side_adata = prepare_data_for_side_by_side_2d(
    adata_dict=daily_adata_dict,
    time_keys=time_keys,
    spatial_key="spatial",
    spacing=100,
    target_size=1000,
    label_to_color=label_to_color,
)

side_by_side_path = SIDE_DIR / "2D_side_by_side_daily_formal.svg"
plot_side_by_side_spatial_with_custom_colors(
    side_by_side_adata,
    color_by="celltype_prediction",
    time_key="original_timepoint",
    spot_size=12,
    figsize=(24, 9),
    save_path=side_by_side_path,
)

stackbar_path = SIDE_DIR / "celltype_stackbar_daily_formal.svg"
plot_celltype_stackbar(
    adata_dict=daily_adata_dict,
    time_keys=time_keys,
    label_to_color=label_to_color,
    annotation_key="celltype_prediction",
    save_path=stackbar_path,
)


In [ ]:
loaded = cb.tl.load_dynamical_model_from_dir(
    MODEL_DIR,
    dim=52,
    device=DEVICE,
    edge_predictor_path=EDGE_PREDICTOR_PATH,
)
runtime = cb.tl.build_dynamical_runtime(loaded)
model = loaded.model
interaction_threshold = float(getattr(getattr(model, "interaction_net", None), "cutoff", 1000.0))

velocity_components_by_time: dict[str, dict[str, np.ndarray]] = {}
for time_label in time_keys:
    adata_t = daily_adata_dict[time_label]
    model_time = float(label_to_model_time[time_label])
    components = cb.tl.compute_velocity_components(
        data=np.asarray(adata_t.X, dtype=np.float32),
        time_value=model_time,
        model=model,
        interaction_m=1024,
        interaction_threshold=interaction_threshold,
        device=DEVICE,
        spatial_dim=2,
    )
    velocity_components_by_time[time_label] = components

print(f"Loaded formal weight stage: {loaded.weight_stage}")
print(f"Loaded formal score stage: {loaded.score_stage}")
print(f"Interaction cutoff: {interaction_threshold}")
print(f"Velocity components computed for {len(velocity_components_by_time)} timepoints.")

In [ ]:
def plot_daily_velocity_stream_grid(
    adata_dict: dict[str, ad.AnnData],
    velocity_by_time: dict[str, dict[str, np.ndarray]],
    palette: dict[str, str],
    color_key: str,
    component_key: str,
    save_path: Path,
    n_cols: int = 4,
):
    ordered = list(adata_dict.keys())
    n_panels = len(ordered)
    n_rows = int(np.ceil(n_panels / n_cols))

    all_coords = np.concatenate([np.asarray(adata_dict[key].obsm["spatial"], dtype=float) for key in ordered], axis=0)
    x_min, y_min = all_coords.min(axis=0)
    x_max, y_max = all_coords.max(axis=0)
    pad_x = (x_max - x_min) * 0.05
    pad_y = (y_max - y_min) * 0.05
    global_xlim = [x_min - pad_x, x_max + pad_x]
    global_ylim = [y_min - pad_y, y_max + pad_y]

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 6 * n_rows))
    axes = np.atleast_1d(axes).ravel()

    present_labels: list[str] = []
    for ax, time_label in zip(axes, ordered):
        adata_t = adata_dict[time_label]
        coords = np.asarray(adata_t.obsm["spatial"], dtype=np.float32)
        if color_key not in adata_t.obs.columns:
            raise KeyError(f"Velocity coloring requires obs['{color_key}'] for {time_label}.")
        labels = pd.Series(adata_t.obs[color_key].astype(str).map(canonical_label), index=adata_t.obs_names)
        for label in labels.unique().tolist():
            if label not in present_labels:
                present_labels.append(label)

        velocity_high_dim = np.asarray(velocity_by_time[time_label][component_key], dtype=np.float32)
        gene_data = np.asarray(adata_t.X[:, 2:], dtype=np.float32)
        gene_velocity = np.asarray(velocity_high_dim[:, 2:], dtype=np.float32)
        ad_plot = ad.AnnData(X=gene_data)
        ad_plot.obsm["X_spatial"] = coords.copy()
        ad_plot.layers["Ms"] = gene_data.copy()
        ad_plot.layers["velocity"] = gene_velocity.copy()
        ad_plot.obs[color_key] = labels.values
        ad_plot.obs[color_key] = ad_plot.obs[color_key].astype("category")
        cats = list(ad_plot.obs[color_key].cat.categories)
        ad_plot.uns[f"{color_key}_colors"] = [palette.get(cat, "#888888") for cat in cats]
        # Formal native gene-space rendering builds the transition graph in
        # model state space (X), then projects that velocity field back to
        # the observed spatial coordinates for stream plotting.
        sc.pp.neighbors(ad_plot, n_neighbors=30, use_rep="X")
        scv.tl.velocity_graph(ad_plot, vkey="velocity", xkey="Ms", n_jobs=1)
        scv.tl.velocity_embedding(ad_plot, basis="spatial", vkey="velocity")

        scv.pl.velocity_embedding_stream(
            ad_plot,
            basis="spatial",
            color=color_key,
            ax=ax,
            show=False,
            legend_loc="none",
            size=160,
            density=4.8,
            alpha=0.8,
            linewidth=0.45,
            arrowsize=0.6,
            min_mass=0.01,
            smooth=0.2,
            title=time_label,
        )
        ax.set_xlim(global_xlim)
        ax.set_ylim(global_ylim)
        ax.set_aspect("equal", adjustable="box")

    for ax in axes[n_panels:]:
        ax.axis("off")

    legend_handles = [mpatches.Patch(color=palette[label], label=label) for label in present_labels if label in palette]
    fig.legend(handles=legend_handles, title=color_key, loc="center left", bbox_to_anchor=(0.92, 0.5), frameon=False, labelspacing=1.25)
    fig.suptitle(f"Formal velocity: {component_key} (gene space, colored by {color_key})", fontsize=16)
    plt.tight_layout(rect=[0, 0, 0.9, 0.96])
    fig.savefig(save_path, dpi=300, bbox_inches="tight")
    print(f"Saved velocity stream grid to {save_path}")
    plt.show()
    return fig


observed_velocity_adata_dict = {key: daily_adata_dict[key] for key in observed_time_keys}
velocity_outputs = []
for component_key in ["full", "drift", "interaction"]:
    out_path = VELOCITY_DIR / f"velocity_stream_{component_key}_gene_observed_region_formal.svg"
    plot_daily_velocity_stream_grid(
        adata_dict=observed_velocity_adata_dict,
        velocity_by_time=velocity_components_by_time,
        palette=region_to_color,
        color_key="region",
        component_key=component_key,
        save_path=out_path,
        n_cols=4,
    )
    velocity_outputs.append(str(out_path))

In [ ]:
cached_classifier = cb.tl.load_cached_mlp_classifier(str(CLASSIFIER_CACHE_PATH), device=DEVICE)


def compute_transition_matrix(source_labels: np.ndarray, target_labels: np.ndarray) -> pd.DataFrame:
    all_types = sorted({str(v) for v in source_labels} | {str(v) for v in target_labels})
    matrix = pd.DataFrame(0.0, index=all_types, columns=all_types)
    for src, tgt in zip(source_labels, target_labels):
        matrix.loc[str(src), str(tgt)] += 1.0
    row_sums = matrix.sum(axis=1)
    return matrix.div(row_sums.replace(0, np.nan), axis=0).fillna(0.0)


def integrate_formal_forward_interval(
    x0: np.ndarray,
    t_start: float,
    t_end: float,
    dt: float = 0.05,
) -> np.ndarray:
    x = np.asarray(x0, dtype=np.float32).copy()
    if np.isclose(t_start, t_end):
        return x
    total = float(t_end - t_start)
    n_steps = max(1, int(np.ceil(abs(total) / dt)))
    step_dt = total / n_steps
    t_value = float(t_start)
    for _ in range(n_steps):
        velocity = cb.tl.compute_velocity_components(
            data=x,
            time_value=t_value,
            model=model,
            interaction_m=1024,
            interaction_threshold=interaction_threshold,
            device=DEVICE,
            spatial_dim=2,
        )["full"]
        x = x + np.asarray(velocity, dtype=np.float32) * np.float32(step_dt)
        t_value += step_dt
    return x


def build_transition_matrices(sequence_keys: list[str]) -> dict[tuple[str, str], pd.DataFrame]:
    matrices: dict[tuple[str, str], pd.DataFrame] = {}
    for src_label, tgt_label in zip(sequence_keys[:-1], sequence_keys[1:]):
        src_time = float(label_to_model_time[src_label])
        tgt_time = float(label_to_model_time[tgt_label])
        source_adata = daily_adata_dict[src_label]
        source_points = np.asarray(source_adata.X, dtype=np.float32)
        source_labels = source_adata.obs["celltype_prediction"].astype(str).map(canonical_label).to_numpy()
        simulated_points = integrate_formal_forward_interval(source_points, src_time, tgt_time)
        predicted_labels = cb.tl.predict_labels_for_points(
            points=simulated_points,
            time_value=tgt_time,
            model=cached_classifier.model,
            label_encoder=cached_classifier.label_encoder,
            feature_dim=cached_classifier.feature_dim,
            device=DEVICE,
            knn_neighbors=1,
            include_time_feature=cached_classifier.include_time_feature,
            spatial_coords=simulated_points[:, :2],
        )
        matrices[(src_label, tgt_label)] = compute_transition_matrix(source_labels, predicted_labels)
    return matrices


def build_lineage_adata(sequence_keys: list[str], gap_scale: float = 0.2, space: float = 0.2) -> ad.AnnData:
    return prepare_multi_timepoint_adata(
        data_list=[np.asarray(daily_adata_dict[key].X, dtype=np.float32) for key in sequence_keys],
        labels_list=[daily_adata_dict[key].obs["celltype_prediction"].astype(str).map(canonical_label).to_numpy() for key in sequence_keys],
        timepoint_labels=sequence_keys,
        gap_scale=gap_scale,
        space=space,
        align_bottom=True,
    )


def render_lineage(
    sequence_keys: list[str],
    save_name: str,
    *,
    step_configs: dict[tuple[str, str], dict[str, object]] | None = None,
    figsize: tuple[float, float] = (30, 6),
    fixed_ct_order: dict[str, int] | None = None,
) -> dict[str, object]:
    transition_matrices = build_transition_matrices(sequence_keys)
    lineage_adata = build_lineage_adata(sequence_keys)
    svg_path = LINEAGE_DIR / save_name
    plot_lineage_transition(
        adata_combined=lineage_adata,
        transition_matrices=transition_matrices,
        time_keys=sequence_keys,
        label_to_color=label_to_color,
        step_configs=step_configs,
        figsize=figsize,
        fixed_ct_order=fixed_ct_order,
        save_path=str(svg_path),
    )

    matrix_dir = LINEAGE_DIR / svg_path.stem
    matrix_dir.mkdir(parents=True, exist_ok=True)
    for (src_label, tgt_label), matrix in transition_matrices.items():
        matrix.to_csv(matrix_dir / f"transition_matrix_{src_label}_to_{tgt_label}.csv")

    print(f"Saved lineage SVG to {svg_path}")
    print(f"Saved transition matrices to {matrix_dir}")
    display(SVG(filename=str(svg_path)))
    return {"svg": str(svg_path), "matrix_dir": str(matrix_dir)}


full_daily_lineage = render_lineage(
    sequence_keys=time_keys,
    save_name="lineage_transition_D4_to_D14_daily_formal.svg",
    step_configs={
        (src_label, tgt_label): {
            "focus_source": ["TMSB4X high cells"],
            "focus_target": ["TMSB4X high cells"],
            "exclude": ["Erythrocytes", "Macrophages"],
        }
        for src_label, tgt_label in zip(time_keys[:-1], time_keys[1:])
    },
    figsize=(30, 6),
)

cardiomyocyte_step_configs = {
    ("D4", "D7"): {
        "exclude": sorted(
            set(label_to_color)
            - {"Cardiomyocytes-1", "Immature myocardial cells"}
        ),
        "focus_source": ["Immature myocardial cells"],
        "focus_target": ["Cardiomyocytes-1", "Immature myocardial cells"],
    },
    ("D7", "D10"): {
        "exclude": sorted(
            set(label_to_color)
            - {"Cardiomyocytes-1", "Immature myocardial cells"}
        ),
        "focus_source": ["Cardiomyocytes-1", "Immature myocardial cells"],
        "focus_target": ["Cardiomyocytes-1", "Immature myocardial cells"],
    },
    ("D10", "D14"): {
        "exclude": sorted(
            set(label_to_color)
            - {"Cardiomyocytes-1", "Immature myocardial cells"}
        ),
        "focus_source": ["Cardiomyocytes-1", "Immature myocardial cells"],
        "focus_target": ["Cardiomyocytes-1", "Immature myocardial cells"],
    },
}
cardiomyocyte_fixed_order = {
    "Immature myocardial cells": 0,
    "Cardiomyocytes-1": 1,
}
cardiomyocyte_lineage = render_lineage(
    sequence_keys=["D4", "D7", "D10", "D14"],
    save_name="lineage_transition_D4_D7_D10_D14_cardiomyocytes_formal.svg",
    step_configs=cardiomyocyte_step_configs,
    figsize=(12, 6),
    fixed_ct_order=cardiomyocyte_fixed_order,
)

epi_to_fib_step_configs = {
    ("D10", "D12"): {
        "focus_source": ["Epi-epithelial cells"],
        "focus_target": [],
        "exclude": [
            "Erythrocytes", "Macrophages", "Endocardial cells", "Vascular endothelial cells",
            "TMSB4X high cells", "Cardiomyocytes-1", "Immature myocardial cells",
            "MT-enriched cardiomyocytes", "Cardiomyocytes-2", "Mural cells",
        ],
    },
    ("D12", "D14"): {
        "focus_source": [],
        "focus_target": ["Fibroblast cells"],
        "exclude": [
            "Erythrocytes", "Macrophages", "Endocardial cells", "Vascular endothelial cells",
            "TMSB4X high cells", "Mural cells", "Cardiomyocytes-1", "Immature myocardial cells",
            "MT-enriched cardiomyocytes", "Cardiomyocytes-2",
        ],
    },
}
epi_to_fib_lineage = render_lineage(
    sequence_keys=["D10", "D12", "D14"],
    save_name="lineage_transition_D10_D12_D14_epi_to_fibroblast_formal.svg",
    step_configs=epi_to_fib_step_configs,
    figsize=(12, 6),
)

In [ ]:
COMM_EXCLUDE_CELLTYPES = [
    "Erythrocytes",
    "Macrophages",
    "Cardiomyocytes-2",
    "MT-enriched cardiomyocytes",
    "TMSB4X high cells",
]

formal_comm_adata_dict = {
    str(float(label_to_model_time[label])): adata_t.copy()
    for label, adata_t in daily_comm_adata_dict.items()
}
formal_comm_time_points = [float(label_to_model_time[label]) for label in time_keys]

stderr_buffer = io.StringIO()
with redirect_stderr(stderr_buffer):
    with plt.ioff():
        formal_all_time_communications = cb.tl.compute_timepoint_communications(
            adata_dict=formal_comm_adata_dict,
            time_points=formal_comm_time_points,
            annotation_key="celltype_prediction",
            f_net=runtime.f_net,
            device=DEVICE,
            out_dir=str(COMMUNICATION_DIR / "attention"),
            remove_self_loop=False,
            winsor_quantile=0.995,
            save_pickle_path=str(COMMUNICATION_DIR / "all_time_communications.pkl"),
        )
        all_time_communications = {
            label: formal_all_time_communications[str(float(label_to_model_time[label]))]
            for label in time_keys
        }
        plt.close("all")

        network_svg_path = plot_segment_network_evolution(
            segment_name="Global_Evolution_D4_to_D14",
            segment_labels=time_keys,
            adata_dict=daily_adata_dict,
            all_time_communications=all_time_communications,
            label_to_color=label_to_color,
            exclude_types=COMM_EXCLUDE_CELLTYPES,
            output_dir=COMMUNICATION_DIR,
        )
        plt.close("all")

print(f"Saved communication network SVG to {network_svg_path}")
network_svg_text = Path(network_svg_path).read_text(encoding="utf-8")
display(SVG(data=network_svg_text))

In [ ]:
summary = {
    "input_run_dir": str(INTERP_RUN_DIR),
    "model_dir": str(MODEL_DIR),
    "device": DEVICE,
    "time_keys": time_keys,
    "observed_time_keys": observed_time_keys,
    "label_to_model_time": label_to_model_time,
    "palette": label_to_color,
    "region_palette": region_to_color,
    "outputs": {
        "side_by_side": str(side_by_side_path),
        "stackbar": str(stackbar_path),
        "velocity_streams": velocity_outputs,
        "lineage_full_daily": full_daily_lineage,
        "lineage_cardiomyocyte": cardiomyocyte_lineage,
        "lineage_epi_to_fibroblast": epi_to_fib_lineage,
        "communication_network_svg": str(network_svg_path),
        "combined_timeline_h5ad": str(SAVED_ANDATA_DIR / "daily_real_plus_interpolated_all_times.h5ad"),
        "combined_timeline_csv": str(SAVED_ANDATA_DIR / "daily_real_plus_interpolated_all_times.csv"),
    },
}

summary_path = OUTPUT_DIR / "replot_summary.json"
with summary_path.open("w", encoding="utf-8") as handle:
    json.dump(summary, handle, indent=2)

print(f"Saved replot summary to {summary_path}")
print(f"Saved side-by-side plot to {side_by_side_path}")
print(f"Saved stackbar plot to {stackbar_path}")
print(f"Saved communication network to {network_svg_path}")
for path in velocity_outputs:
    print(path)